# **Parallel Chain**

In [16]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableBranch
from pydantic import BaseModel
from typing import Literal

load_dotenv()


True

In [17]:
#Base Model

promt_template = ChatPromptTemplate.from_messages([
    ("system","you are a movie reviewer"),
    ("human","Write some review on the movie : {name}")
])

llm = ChatOpenAI(model="gpt-5-mini",temperature=0)


str_parser= StrOutputParser()


## Parallel Chain 1

In [18]:
facebbok_prompt = ChatPromptTemplate.from_messages([
    ("system","you are a funny writter who will create funny post for Facebook"),
    ("human","create a facebook post for the following text : {text}")

])

chain_facebook = facebbok_prompt | llm | str_parser

## Parallel Chain 2

In [19]:
def x_chain(input: dict):
    x_prompt = ChatPromptTemplate.from_messages([
        ("system","you are a serious content writter on x platform"),
        ("human","create a serious post for the following text : {text}")
    ])

    chain_x = x_prompt | llm | str_parser

    return chain_x.invoke(input)   # ✅ directly pass input

x_chain_runnable = RunnableLambda(x_chain)

## Conditional Chain

In [20]:


class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive","negative"]

llm_structured_output=llm.with_structured_output(llm_schema)

# ressss = llm_structured_output.invoke("The movie was grate")
# ressss

In [21]:


def dic_maker(input:llm_schema) ->  str:
    return input.model_dump()["movie_summary_flag"]

dic_maker_runnable = RunnableLambda(dic_maker)

In [22]:
conditional_chain = RunnableBranch(
    (lambda x: x == "positive", chain_facebook),
    x_chain_runnable
)

## Final Conditonal Chain Orchestration

In [23]:
final_conditional_chain = promt_template | llm_structured_output | dic_maker_runnable | conditional_chain

In [24]:
final_response = final_conditional_chain.invoke({"name": "I love Baashha"})  

In [27]:
final_response


'Official announcement: I\'ve legally changed my mood to "Positive." 🚀✨\n\nBenefits include:\n- Free smiles with every breath 😄\n- Unlimited patience (terms and conditions may apply) 🕊️\n- Immediate immunity to bad vibes (scientifically unproven but highly effective) 🛡️\n\nPS — If you see me walking around radiating optimism, don\'t be alarmed. It\'s just my natural state. Feel free to catch some—it\'s contagious! 😂\n\nDrop a "+" in the comments if you\'re joining the positive squad. 💪🌈'